# Hyperparameter Tuning: Grid Search, Randomized Search, and Bergstra-Bengio Theory
Accompanies **Unit 5.3: Cross-Validation & Hyperparameter Tuning**.

In high-dimensional hyperparameter spaces, Grid Search wastes exponential computing power evaluating permutations of parameters that have zero real influence on performance.

The **Bergstra-Bengio Theorem (2012)** proves that testing $n$ randomly drawn combinations gives a probability:
$$P = 1 - (1 - p)^n$$
of capturing a configuration in the top $p$-fraction of the true parameter space. 
For $p = 0.05$ (top $5\%$), testing just $n = 60$ random points yields:
$$P = 1 - (1 - 0.05)^{60} = 1 - (0.95)^{60} \approx 95.4\%$$
completely independent of the total dimension of the hyperparameter space!

In this notebook, we compare Grid Search against Randomized Search inside leak-free scikit-learn pipelines.

In [ ]:
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.datasets import make_classification
from scipy.stats import randint

X, y = make_classification(n_samples=1500, n_features=25, n_informative=8, random_state=42)

# Grid Search space: 3 * 3 * 3 * 3 = 81 evaluations
grid_params = {
    'n_estimators': [50, 100, 150],
    'max_depth': [4, 8, 12],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Measure Grid Search time
start = time.time()
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid=grid_params, cv=3, n_jobs=-1)
grid.fit(X, y)
grid_time = time.time() - start

# Randomized Search space: continuous distributions, testing 30 iterations
rand_params = {
    'n_estimators': randint(50, 200),
    'max_depth': randint(3, 16),
    'min_samples_split': randint(2, 12),
    'min_samples_leaf': randint(1, 8)
}

start = time.time()
rand_search = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_distributions=rand_params, n_iter=30, cv=3, random_state=42, n_jobs=-1)
rand_search.fit(X, y)
rand_time = time.time() - start

print(f"Grid Search (81 models):       {grid_time:.2f}s | Best Accuracy: {grid.best_score_:.4f}")
print(f"Randomized Search (30 models): {rand_time:.2f}s | Best Accuracy: {rand_search.best_score_:.4f}")
print(f"Speedup Factor: {grid_time / rand_time:.2f}x faster with comparable or superior performance!")